# **NOTICE**
Not all code will be required for every assignment, exam, project, etc. Select only what is needed for the task.

# **IMPORTS**

In [ ]:
###----DEPENDENCIES/METRICS/RANDOM----###

#--NOT PYSPARK--#
# DMBA Installation
!pip install dmba
# Classification Results
from dmba import classificationSummary
from sklearn.metrics import classification_report
# Parameter Adjustments
from sklearn.model_selection import GridSearchCV
# Regression Summary
from dmba import regressionSummary
# Coefficients and P-Values (Statistical Modeling)
import statsmodels.formula.api as smf
# Plot Trees
from dmba import plotDecisionTree
# Google Colab Mounting
from google.colab import drive

#--PYSPARK--#
# Install PySpark
!pip install pyspark
# Spark Session
from pyspark.sql import SparkSession
# Vector Assembler
from pyspark.ml.feature import VectorAssembler
# Metrics
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# Dataframe Editing
from pyspark.sql.functions import (col, explode, array, lit, udf)
# Scaling
from pyspark.ml.feature import MinMaxScaler
from pyspark.ml.feature import StandardScaler
# String Indexer
from pyspark.ml.feature import StringIndexer

###----DATA----###

# Importing Data
import pandas as pd
# Splitting Data
from sklearn.model_selection import train_test_split
# Oversampling (Binary and Multi-Class Output)
from imblearn.over_sampling import SMOTE
# Undersampling (Binary and Muti-Class Output)
from imblearn.under_sampling import RandomUnderSampler
# Scaling
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# Normal Distribution
from scipy.stats import norm

###----CLASSIFICATION----###

#--Sklearn--#
# Logistic Regression (Binary and Multinomial)
from sklearn.linear_model import LogisticRegression as skLGR
# Decision Tree
from sklearn.tree import DecisionTreeClassifier as skDTC
# K-Nearest Neighbors Classifier
from sklearn.neighbors import KNeighborsClassifier as skKNC
# Bernoulli Naive-Bayes
from sklearn.naive_bayes import BernoulliNB as skBNB
# Gaussian Naive-Bayes
from sklearn.naive_bayes import GaussianNB as skGNB
# Multinomial Naive-Bayes
from sklearn.naive_bayes import MultinomialNB as skMNB
# Discriminant Analysis
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as skLDA
# Support Vector Machines (Classification)
from sklearn.svm import SVC as skSVC
# Neural Network (MLP, Classification)
from sklearn.neural_network import MLPClassifier as skNNC
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier as skRFC

#--PySpark--#
# Logistic Regression
from pyspark.ml.classification import LogisticRegression as psLGR
# Decision Tree
from pyspark.ml.classification import DecisionTreeClassifier as psDTC
# Naive-Bayes
from pyspark.ml.classification import NaiveBayes as psNB
# Support Vector Machines
from pyspark.ml.classification import LinearSVC as psSVC
# Neural Network (MLP, Classification)
from pyspark.ml.classification import MultilayerPerceptronClassifier as psNNC
# Random Forest Classifier
from pyspark.ml.classification import RandomForestClassifier as psRFC

###----REGRESSION----###

#--Sklearn--#
# Linear Regression
from sklearn.linear_model import LinearRegression as skLNR
# Regression Tree
from sklearn.tree import DecisionTreeRegressor as skDTR
# K-Nearest Neighbors Regressor
from sklearn.neighbors import KNeighborsRegressor as skKNR
# Support Vector Machines (Regression)
from sklearn.svm import SVR as skSVR
# Neural Network (MLP, Regression)
from sklearn.neural_network import MLPRegressor as skNNR
# Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor as skRFR

#--PySpark--#
# Linear Regression
from pyspark.ml.regression import LinearRegression as psLNR
# Regression Tree
from pyspark.ml.regression import DecisionTreeRegressor as psDTR
# Random Forest Regressor
from pyspark.ml.regression import RandomForestRegressor as psRFR

###----RECOMMENDATION SYSTEMS----###

#--ALS--#
from pyspark.ml.recommendation import ALS

#--Association Rules--#
from pyspark.sql import functions as F
from pyspark.ml.fpm import FPGrowth

###----TEXT PREPROCESSING----###

# Tokenization - Regex better, no code provided for Tokenizer
from pyspark.ml.feature import Tokenizer, RegexTokenizer
#Unk
from pyspark.sql.types import IntegerType
# Stop Word Removal
from pyspark.ml.feature import StopWordsRemover
# Count Vectorization
from pyspark.ml.feature import CountVectorizer
# TF-IDF
from pyspark.ml.feature import HashingTF, IDF
# N-Grams - Used infrequently, no code provided
from pyspark.ml.feature import NGram
# Vectorization - Used infrequently, no code provided
from pyspark.ml.feature import Word2Vec

###----ENSEMBLE CREATION----###

#--Regression--#
# Bagging
from sklearn.ensemble import BaggingRegressor
# Boosting
from sklearn.ensemble import AdaBoostRegressor

#--Classification--#
# Bagging
from sklearn.ensemble import BaggingClassifier
# Boosting
from sklearn.ensemble import AdaBoostClassifier

# **DATA PREPARATION**

## **Sklearn/Python**

In [ ]:
###----STANDARD----###

#--Direct--#
# Data import
df_pd = pd.read_csv('data.csv')

#--GDrive--#
# Mount Google Drive
drive.mount('/content/drive')
# Data import
df_pd = pd.read_csv('/content/drive/My Drive/data.csv')

#--Cleaning and Basic EDA--#
# Data information
df_pd.info()
df_pd.head()
df_pd.describe()
# Correct data type(s)
df_pd['column'] = df_pd['column'].astype('type')
# Common Visualizations
df_pd.hist()
df_pd.boxplot()
# Impute data
df_pd['column'] = df_pd['column'].fillna()
df_pd['column'] = df_pd['column'].fillna(df_pd['column'].aggfunct())
# Value counts and possible entries
df_pd.value_counts()
df_pd.unique()

###----ML PREPARATIONS----###

#--General--#
# Create dummy variables
columns = ['column_a', 'column_b']
df_pd = pd.get_dummies(df_pd, columns=columns, drop_first=True)
# Split data
features = ['feature_a', 'feature_b']
Xtrain_pd, Xtest_pd, ytrain_pd, ytest_pd = train_test_split(df_pd.drop('y'),
                                                            df_pd['y'],
                                                            test_size=0.2,
                                                            random_state=0)

#--Sampling Adjustment, Binary Output--#
# Oversample the data
Xtrain_pd_os, ytrain_pd_os = SMOTE().fit_resample(Xtrain_pd, ytrain_pd)
# Undersample the data
Xtrain_pd_us, ytrain_pd_us = RandomUnderSampler(sampling_strategy='majority').fit_resample(Xtrain_pd, ytrain_pd)

#--Sampling Adjustment, Multiclass Output--#
# Oversample the data
Xtrain_pd_os, ytrain_pd_os = SMOTE().fit_resample(Xtrain_pd, ytrain_pd)
# Undersample the data
Xtrain_pd_us, ytrain_pd_us = RandomUnderSampler().fit_resample(Xtrain_pd, ytrain_pd)

#--Scaling, Standard Scaler--#
# Scale Xtrains
Xtrain_pd_sts = StandardScaler().fit_transform(Xtrain_pd)
Xtrain_pd_os_sts = StandardScaler().fit_transform(Xtrain_pd_os)
Xtrain_pd_us_sts = StandardScaler().fit_transform(Xtrain_pd_us)
# Scale Xtest
Xtest_pd_sts = StandardScaler().fit_transform(Xtest_pd)
# Confirm data
Xtrain_pd_sts.info()
Xtrain_pd_os_sts.info()
Xtrain_pd_us_sts.info()
Xtest_pd_sts.info()

#--Scaling, MinMaxScaler--#
# Scale Xtrains
Xtrain_pd_mms = MinMaxScaler().fit_transform(Xtrain_pd)
Xtrain_pd_os_mms = MinMaxScaler().fit_transform(Xtrain_pd_os)
Xtrain_pd_us_mms = MinMaxScaler().fit_transform(Xtrain_pd_us)
# Scale Xtest
Xtest_pd_mms = MinMaxScaler().fit_transform(Xtest_pd)
# Confirm data
Xtrain_pd_mms.info()
Xtrain_pd_os_mms.info()
Xtrain_pd_us_mms.info()
Xtest_pd_mms.info()

#--Normal Distribution--#
# Normally distribute Xtrains
Xtrain_pd_nm = norm.cdf(Xtrain_pd)
Xtrain_pd_os_nm = norm.cdf(Xtrain_pd_os)
Xtrain_pd_us_nm = norm.cdf(Xtrain_pd_us)
# Normally distribute Xtests
Xtest_pd_nm = norm.cdf(Xtest_pd)
# Confirm data
Xtrain_pd_nm.info()
Xtrain_pd_os_nm.info()
Xtrain_pd_us_nm.info()
Xtest_pd_nm.info()

## **PySpark**

In [ ]:
###----STANDARD----###
#--PySpark Setup--#
# Start Spark session
spark = SparkSession.builder.appName('name').getOrCreate()
# Mount to GDrive
drive.mount('/content/drive')
# Define GDrive path
path = '/content/drive/My Drive/path'

#--Data Reading, Cleaning, and Basic EDA--#
# Read data
df_ps = spark.read.csv(path, header=True, inferSchema=True)
# Show data
df_ps.show()
# Drop column
df_ps = df_ps.drop('column')
# Get value counts
df_ps.groupBy('column').count().show()
# Drop rows with null value(s)
df_ps = df_ps.dropna(how='any')

###----PREPROCESS TEXT----###

#--Tokenization--#
# Define Regex tokenizer
rgx = RegexTokenizer(inputCol='column', outputCol='tokens', pattern='\\W')
# Tokenize data
df_ps = rgx.transform(df_ps)

#--Stopword Removal--#
# Define stopword remover
nstp = StopWordsRemover(inputCol='tokens', outputCol='filtered',caseSensitive=False)
# Remove stopwords
df_ps = nstp.transform(df_ps)

#--Term Frequency--#
# Define TF calculator
htf = HashingTF(inputCol='filtered', outputCol='tf')
# Calculate TF
df_ps = htf.transform(df_ps)

#--TF-IDF--#
# Define IDF calculator
tfidf = IDF(inputCol='tf', outputCol='tfidf')
# Calculate TF-IDF
df_ps = tfidf.fit(df_ps).transform(df_ps)

###----SUPERVISED ML PREPARATIONS----###

#--General--#
# Dummy variables - verify that this works; not taught in class
df_ps = spark.get_dummies(df_ps, drop_first=True)
# String indexing
indexer=StringIndexer(inputCol='y', outputCol='y_index')
indexerModel=indexer.fit(df_ps)
df_ps=indexerModel.transform(df_ps)
# Vector assembly
assembler=VectorAssembler(inputCols=['feature_a', 'feature_b'], outputCol='features')
df_ps=assembler.transform(df_ps)
# Split data
train_ps, test_ps = df_ps.randomSplit([0.8, 0.2], seed=0)

#--Sampling Adjustment, Binary Output--#
# Find classes
major_bn = df_ps.filter(df_ps['y_index'] == 0).count() # 0 is a placeholder for the largest class
minor_bn = df_ps.filter(df_ps['y_index'] == 1).count() # 1 is a placeholder for the largest class
# Define ratio
ratio = int(major_bn/minor_bn)
# Oversample the data
os = minor_bn.withColumn('dummy', explode(array([lit(i) for i in range(ratio)]))).drop('dummy')
train_ps_os = train_ps.unionAll(os)
# Undersample the data
us = major_bn.sample(False, 1/ratio)
train_ps_us = us.unionAll(minor_bn)

#--Sampling Adjustment, Multiclass Output--#
# Find classes
size_0 = df_ps.filter(df_ps['y_index'] == 0).count() # Pretend this is the largest class
size_1 = df_ps.filter(df_ps['y_index'] == 1).count()
size_2 = df_ps.filter(df_ps['y_index'] == 2).count() # Pretend this is the smallest class
# Define ratios
ratio_1 = int(size_0/size_1)
ratio_2 = int(size_0/size_2)
# Oversample the data
os_1 = size_1.withColumn('dummy', explode(array([lit(i) for i in range(ratio_1)]))).drop('dummy')
os_2 = size_2.withColumn('dummy', explode(array([lit(i) for i in range(ratio_2)]))).drop('dummy')
train_ps_os = train_ps_os.unionAll(os_1)
train_ps_os = train_ps_os.unionAll(os_2)
# Undersample the data
ratio_0 = int(size_0/size_2)
ratio_1 = int(size_1/size_2)
us_0 = size_0.sample(False, 1/ratio_0)
us_1 = train_ps_us.sample(False, 1/ratio_1)
train_ps_us = size_2.unionAll(us_0)
train_ps_us = train_ps_us.unionAll(us_1)

#--Scaling, Standard Scaler--# - Verify; Not Used In Class
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)
train_ps_sts = scaler.fit(train_ps).transform(train_ps)
train_ps_os_sts = scaler.fit(train_ps_os).transform(train_ps_os)
train_ps_us_sts = scaler.fit(train_ps_us).transform(train_ps_us)
test_ps_sts = scaler.fit(test_ps).transform(test_ps)

#--Confirm Data--#
train_ps_sts.show()
train_ps_os_sts.show()
train_ps_us_sts.show()
test_ps_sts.show()

#--Scaling, MinMaxScaler--#
scaler = MinMaxScaler(inputCol="features", outputCol="scaledFeatures")
train_ps_mms = scaler.fit(train_ps).transform(train_ps)
train_ps_os_mms = scaler.fit(train_ps_os).transform(train_ps_os)
train_ps_us_mms = scaler.fit(train_ps_us).transform(train_ps_us)
test_ps_mms = scaler.fit(test_ps).transform(test_ps)

#--Confirm Data--#
train_ps_mms.show()
train_ps_os_mms.show()
train_ps_us_mms.show()
test_ps_mms.show()

#--Normally Distribute--# Not Covered In Class

# **SUPERVISED ML**

## **Regression**
Produce a continuous output.

### **SKLEARN**

#### **Scaling Not Required**

##### **CUSTOM FUNCTION**

In [ ]:
# Custom Function for Non-Neural-Net Models
def skreg(mdl,xtr,xts,ytr,yts):
  mdl.fit(xtr,ytr) # Fit model to training data
  trpreds = mdl.predict(xtr) # Predict y values for xtr
  tspreds = mdl.predict(xts) # Predict y values for xts
  print('Training')
  print(regressionSummary(ytr,trpreds)) # Regression summary for training data
  print('')
  print('Testing')
  print(regressionSummary(yts,tspreds)) # Regression summary for testing data

# Example of Custom Function Usage
skreg(skLNR(fit_intercept=False), Xtrain_pd, Xtest_pd, ytrain_pd, ytest_pd)

##### **LINEAR REGRESSION**
Draws line/plane of best fit through the data space; effectively a multivariate slope function.

In [ ]:
# Main Linear Regression Parameters
skLNR(fit_intercept=False, #Fits an intercept
      positive=False) #Forces coefficients to be positive

##### **REGRESSION TREE**
Analyzes the effect of truth statements about features on the output value.

In [ ]:
# Main Regression Tree Parameters
skDTR(max_depth=None, #Determines max number of decisions in the tree
      min_samples_split=2, #Min samples required to differ when creating a new node
      min_impurity_decrease=0.0) #Degree of impurity reduction produced by a split needed for it to be valid

##### **RANDOM FOREST**
Multiple regression trees running on subsets of the data and converging predicitons at the end.

In [ ]:
# Main Random Forest Parameters
skRFR(n_estimators=100, # How many trees to include in the forest
      max_depth=None,
      min_samples_split=2,
      min_impurity_decrease=0.0)

#### **Scaling Required**

##### **SCALER INFORMATION**
Standard Scaler is used when the data is normally distributed.
Min-Max Scaler is used when the data is not normally distributed.
It is not necessary to scale binary variables, but running them through the scaler should not change them.

In [ ]:
# Standard Scaler
skreg(skKNR(n_estimators=7), Xtrain_pd_sts, Xtest_pd_sts, ytrain_pd, ytest_pd)
# Min-Max Scaler
skreg(skKNR(n_estimators=7), Xtrain_pd_mms, Xtest_pd_mms, ytrain_pd, ytest_pd)

##### **KNN REGRESSOR**
Measures the distance between a datapoint and *k* neighbors, average the target value, and use this mean as the prediction.

In [ ]:
# Main KNN Parameter
skKNR(n_neighbors=5) #Number of neighbors to compare a datapoint to

##### **SVM REGRESSOR**
Uses a kernel to warp the dataspace before drawing a line/plane of best fit, somewhat like linear regression.

In [ ]:
# Main SVM Regressor Parameters
skSVR(kernel='rbf', #Kernel shape
      C=1.0) #Regularization. Higher the number, lower the regularization.

Regularization prevents overfitting by preventing the model from assigning too much weight to any one variable. Basically, penalize - or "regularize" - large coefficients.

#### **Neural Net**

##### **CUSTOM FUNCTION**

In [ ]:
# y Scaling, Standard Scaler
ytrain_pd_sts = StandardScaler().fit_transform(ytrain_pd)
ytest_pd_sts = StandardScaler().fit_transform(ytest_pd)

# y Scaling, Min Max Scaler
ytrain_pd_sts = MinMaxScaler().fit_transform(ytrain_pd)
ytest_pd_sts = MinMaxScaler().fit_transform(ytest_pd)

# Custom Function for Neural-Net Models
def sknn(mdl,xtr,xts,ytr,yts,sclr)
  xscl = sclr # Define scaler for X objects
  yscl = sclr # Define scaler for y objects
  xtr_sc = xscl.fit_transform(xtr) # Fit Xscaler to Xtrain and scale it
  xts_sc = xscl.transform(xts) # Use Xscaler to scale Xtest
  ytr_sc = yscl.fit_transform(ytr) # Fit yscaler to ytrain and scale it
  yts_sc = yscl.transform(yts) # Use yscaler to scale ytest
  mdl.fit(xtr_sc,ytr_sc) # Fit model to scaledtraining data
  trpreds = mdl.predict(xtr_sc) # Predict y values for scaled Xtrain
  tspreds = mdl.predict(xts_sc) # Predict y values for scaled Xtest
  trpreds = yscl.inverse_transform(trpreds) # Inverse scaling of trpreds
  tspreds = yscl.inverse_transform(tspreds) # Inverse scaling of tspreds
  print('Training')
  print(regressionSummary(ytr,trpreds)) # Regression summary for training data
  print('')
  print('Testing')
  print(regressionSummary(yts,tspreds)) # Regression summary for testing data

# Example of Custom Function Usage
sknn(skNNR(), Xtrain_pd, Xtest_pd, ytrain_pd, ytest_pd, StandardScaler())

##### **MLP REGRESSOR**
Feeds entries into neurons which process the data, each connection recieving a random weight (importance). Neurons create predictions which are assigned a random bias. If there is more than one internal neuron layer (hidden layer), the predictions and entries are sent to another layer of neurons, receiving their own weights and biases. This iterates until the model converges.

In [ ]:
skNNR(hidden_layer_sizes=(100,), #Number and size of hidden layers. Here, 1 HL with size 100.
      max_iter=1000, ## of iterations
      activation='relu', #Activation function
      solver='adam', #Weight determinations
      alpha=0.0001, #Regularization
      batch_size='auto', #Size of batches when using 'adam' or 'sgd' solvers
      learning_rate='constant', #Rate at which the model updates weights
      learning_rate_init=0.001, #Where learning rate starts
      early_stopping=False) #If true, stop training when validation score stops improving

There are two main schools of thought for determining hidden layer size:

1. sqrt(*a* * *b*)
2. ((2/3)**a*)+*b*

Where
* a = # of training entries
* b = # of output nodes (1 if regression, # of classes if classification)

The best number of hidden layers depends on the data's complexity, though going beyond two is rare. Hidden layer size should decrease, i.e. (100, 67, 33)

#### **Custom Ensembles**
Bagging and boosting via BaggingRegressor/Classifier and AdaBoostRegressor/Classifier are generally not used with tree-based models or neural networks. For a tree-based boosting model, use gradient boosting or XGBoost (not in sklearn).

##### **BAGGING**
Combines results of several models ran on subsets of the training data.

In [ ]:
skreg(BaggingRegressor(base_estimator=skLNR()),Xtrain, Xtest, ytrain, ytest)

BaggingRegressor(base_estimator=skLNR(), #Model to use
                 n_estimators=100) #Number of subsets to train on

##### **BOOSTING**
Splits data into subsets, trains on one subset, passess incorrectly predicted data entries into the next subset, then trains on the next subset. Iterate until there are no more subsets.

In [ ]:
skreg(BaggingRegressor(base_estimator=skLNR()),Xtrain, Xtest, ytrain, ytest)

AdaBoostRegressor(base_estimator=skLNR(), #Model to use
                  n_estimators=100) # of subsets to train on

### **PYSPARK**

#### **Scaling Not Required**

In [ ]:
def psreg(mdl,trn,tst):
  mdl = mdl
  mod = mdl.fit(trn)
  pred = mod.transform(tst)
  rmse=RegressionEvaluator(predictionCol='prediction',
                           labelCol='y',
                           metricName='rmse')
  print('RMSE: ', rmse.evaluate(pred))
  mae=RegressionEvaluator(predictionCol='prediction',
                           labelCol='y',
                           metricName='mae')
  print('MAE: ', mae.evaluate(pred))
  mse=RegressionEvaluator(predictionCol='prediction',
                           labelCol='y',
                           metricName='mse')
  print('MSE: ', mse.evaluate(pred))
  r2=RegressionEvaluator(predictionCol='prediction',
                           labelCol='y',
                           metricName='r2')
  print('R2: ', r2.evaluate(pred))

psreg(psLNR(),train_ps,test_ps)

#### **Scaling Required**

In [ ]:
psreg(psLNR(),train_ps_sts,test_ps_sts)
psreg(psLNR(),train_ps_mms,test_ps,mms)

## **Classification**
Produce a categorical output.

### **SKLEARN**

#### **Scaling Not Required**

##### **CUSTOM FUNCTION**

In [ ]:
def skcls(mdl,xtr,xts,ytr,yts):
  mdl.fit(xtr,ytr) # Fit model to training data
  trpreds = mdl.predict(xtr) # Predict y values for xtr
  tspreds = mdl.predict(xts) # Predict y values for xts
  print('Training')
  print(classification_report(ytr,trpreds)) # Classification report for training data
  print('')
  print('Testing')
  print(classification_report(yts,tspreds)) # Classification report for testing data

skcls(skLGR(C=1.0), Xtrain_pd, Xtest_pd, ytrain_pd, ytest_pd)

##### **LOGISTIC REGRESSION**
Similar to linear regression, except using log of odds; produces probability that a sample belongs to a given class. By default, >=.50 is labeled as the class.

In [ ]:
# Main Logistic Regression Parameters
skLGR(C=1.0, #Regularization term. Higher the term, lesser the regularization
      multi_class='multinomial', #Determines binary v multi-class classification
      solver='lbfgs', #Solver used in optimzation
      penalty='l2') #Type of regularization to use

##### **CLASSIFICATION TREE**
See regression tree

In [ ]:
skDTC(max_depth=None,
      min_samples_split=2,
      min_impurity_decrease=0.0)

##### **RANDOM FOREST CLASSIFIER**
See random forest regressor

In [ ]:
skRFC(n_estimators=100,
      max_depth=None,
      min_samples_split=2,
      min_impurity_decrease=0.0)

##### **BERNOULLI NAIVE-BAYES**
All Naive-Bayes algorithms work using Bayes' theorem, a method of calculating the probability that an event (in this case, a category) occurs based on previous events.

Bernoulli Naive-Bayes works with binary inputs only.

In [ ]:
skBNB(alpha=1.0, #Smoothing parameter. Higher parameter, greater smoothing
      fit_prior=True) #Learn class prior probabilities

##### **MULTINOMIAL NAIVE-BAYES**
See explanation of Naive-Bayes algorithms provided in Bernoulli Naive-Bayes.

Muntinomial Naive-Bayes works with count data.

In [ ]:
skMNB(alpha=1.0, #Smoothing parameter. Higher parameter, greater smoothing
      fit_prior=True) #Learn class prior probabilities

#### **Scaling Required**

##### **CUSTOM FUNCTION**

In [ ]:
# Standard Scaler
skcls(skKNC(n_estimators=7), Xtrain_pd_sts, Xtest_pd_sts, ytrain_pd, ytest_pd)
# Min-Max Scaler
skcls(skKNC(n_estimators=7), Xtrain_pd_mms, Xtest_pd_mms, ytrain_pd, ytest_pd)

##### **KNN CLASSIFIER**
Similar to KNN Regressor. Instead of averaging the target value, refer to the most common class in the neighborhood (point and its nearest neighbors) and use this mode as the prediction.

In [ ]:
skKNC(n_neighbors=5)

##### **SVM CLASSIFIER**
Similar to SVM Regressor, except use hyperplanes to split the data space into clusters instead of using them as a line of best fit.

In [ ]:
skSVM(kernel='rbf', #Kernel shape
      C=1.0) #Regularization. Higher the number, lower the regularization.

##### **LINEAR DISCRIMINANT ANALYSIS**
Identifies a linear combination of features that optimize classification via Bayes' theorem.

In [ ]:
skLDA(solver='svd', #Decomposer to use
      n_components=None, #of components for dimensionality reduction
      store_covariance=False, #Whether to store the covariance matrix
      tol=0.0001) #Determines which dimensions to use based on perceived significance threshold

##### **MLP CLASSIFIER**
See MLP Regressor explanation.

In [ ]:
skNNC(hidden_layer_sizes=(100,), #Number and size of hidden layers. Here, 1 HL with size 100.
      max_iter=1000, ## of iterations
      activation='relu', #Activation function
      solver='adam', #Weight determinations
      alpha=0.0001, #Regularization
      batch_size='auto', #Size of batches when using 'adam' or 'sgd' solvers
      learning_rate='constant', #Rate at which the model updates weights
      learning_rate_init=0.001, #Where learning rate starts
      early_stopping=False) #If true, stop training when validation score stops improving

#### **Normal Distribution Required**

##### **CUSTOM FUNCTION**

In [ ]:
skcls(skGNB(), Xtrain_pd_nm, Xtest_pd_nm, ytrain_pd, ytest_pd)

##### **GAUSSIAN NAIVE-BAYES**
See explanation of Naive-Bayes algorithms provided in Beroulli Naive-Bayes.

Gaussian Naive-Bayes works with normally distributed data.

In [ ]:
skGNB()

#### **Custom Ensembles**

##### **BAGGING**
See explanation provided in the regression section.

In [ ]:
skcls(BaggingClassifier(base_estimator=skLNR()),Xtrain, Xtest, ytrain, ytest)

BaggingClassifer(base_estimator=skLGR(), #Model to use
                 n_estimators=100) #Number of subsets to train on

##### **BOOSTING**
See explanation provided in the regression section.

In [ ]:
skcls(AdaBoostClassifier(base_estimator=skLNR()),Xtrain, Xtest, ytrain, ytest)

AdaBoostClassifer(base_estimator=skLGR(), #Model to use
                  n_estimators=100) #Number of subsets to train on

### **PYSPARK**

#### **Scaling Not Required**

In [ ]:
def psbcls(mdl,trn,tst):
  mdl = mdl
  mod = mdl.fit(trn)
  pred = mod.transform(tst)
  tp = pred.filter((pred['label']==1) & (pred['prediction']==1)).count()
  fp = pred.filter((pred['label']==0) & (pred['prediction']==1)).count()
  tn = pred.filter((pred['label']==0) & (pred['prediction']==0)).count()
  fn = pred.filter((pred['label']==1) & (pred['prediction']==0)).count()
  acc = ((tp+tn)/(tp+tn+fp+fn))
  prec = (tp/(tp+fp))
  recl = (tp/(tp+fn))
  spec = (tn/(tn+fp))
  print(f'Accuracy: {round(acc,2)}')
  print(f'Precision: {round(prec,2)}')
  print(f'Recall: {round(recl,2)}')
  print(f'Specificity: {round(spec,2)}')

psbcls(psLNR(),train_ps,test_ps)

def psmcls(mdl,trn,tst):
  mdl = mdl
  mod = mdl.fit(trn)
  pred = mod.transform(tst)
  eval1 = MulticlassClassificationEvaluator(labelCol = 'label',
                                          predictionCol = 'prediction',
                                          metricName = 'accuracy')
  accuracy = eval1.evaluate(pred)
  print(f'Accuracy = {round(accuracy, 2)}')
  eval2 = MulticlassClassificationEvaluator(labelCol = 'label',
                                            predictionCol = 'prediction',
                                            metricName = 'precisionByLabel')
  precision = eval2.evaluate(pred)
  print(f'Precision = {round(precision, 2)}')

  eval3 = MulticlassClassificationEvaluator(labelCol = 'label',
                                            predictionCol = 'prediction',
                                            metricName = 'recallByLabel')
  recall = eval3.evaluate(pred)
  print(f'Recall = {round(recall, 2)}')

psmcls(psLNR(),train_ps,test_ps)

#### **Scaling Required**

In [ ]:
psbcls(psLNR(),train_ps_sts,test_ps_sts)
psbcls(psLNR(),train_ps_mms,test_ps_mms)

psmcls(psLNR(),train_ps_sts,test_ps_sts)
psmcls(psLNR(),train_ps_mms,test_ps_mms)

## **Parameter Tuning**

### **SKLEARN**
GridSearchCV will perform cross-validation on provided data, testing all possible combinations of parameters you feed it. Once complete, it will provde the best set of parameters tested along with a metric that shows how well it performed. These metrics are r2 and accuracy for regression and classification respectively.

In class, this is only used for decision and regression trees and is called "pruning", but it can be done for any model.

In [ ]:
# Define Parameters to Test (Tree-Based Models)
params = {'max_depth': [5, 10, 15, 20, 25],
          'min_samples_split': [50, 100, 150, 200, 250],
          'min_impurity_decrease': [0, 0.0001, 0.005, 0.001, 0.01]}

# Custom Function
def sktuning(prms,mdl,xtr,ytr):
  search = GridSearchCV(mdl, prms, cv = 5)
  search.fit(xtr,ytr)
  print(f'Parameters: {search.best_params_}')
  print(f'Score: {search.best_score_}')

# Example of Custom Function Usage
sktuning(params, skDTR(random_state=0), Xtrain_pd, ytrain_pd)

# **UNSUPERVISED ML**

## **Recommendation Systems**

### **Collaborative Filtering via ALS - PySpark Only**
Recommends items based on how others recommend them along with their - and the subjects - preferences for other items. Basically, if two people like the same movies, they're likely to give a similar rating for another movie, so one of them can be recommended the movie if someone else has the same preferences and rated it highly.

In [ ]:
psreg(ALS(),train_ps,test_ps)

### **Association Rules - PySpark Only**
Calculates probabilities of things occurring (consequent) if something else has occurred (antecedent), like someone purchasing eggs if they purchased milk on the same grocery trip.

|transaction|item|
|-|-|
|1|1|
|1|2|
|2|1|

FP prior to aggregation

In [ ]:
# Aggregate items
fp = fp.groupBy('transaction').agg(F.collect_list('item')).sort('transaction')

|transaction|collect_list('item')|
|-|-|
|1|[1,2]|
|2|[1]|

FP after aggregation

In [ ]:
# Generate Association Rules
fpg = FPGrowth(itemsCol='collect_list(item)',minSupport=0.2, #proportion of the dataset containing a given itemset
               minConfidence=0.4) #% likelihood that, given some antecedent(s), a given consequent occurs
mdl = fpg.fit(fp)
mdl.associationRules.show()

# Generate prediciton; what other item(s) may be desired based on the transaction
mdl.transform(fp).show()